# Part B: M3D-LaMed-Phi-3-4B Genuine 3D CT Audit (Zero Dummy)
**Research Question:** *Does M3D-LaMed actually ground its clinical answers in 3D CT image voxels?*

### Strict Zero-Dummy Protocols:
1. **Only Real 3D CT Scans (`.npy`):** Loaded from `medi data 3d` attached dataset.
2. **Fail-Fast Enforcement:** If any CT volume is missing or flat, execution **halts immediately**.
3. **6 Counterfactual Audit Conditions:**
   - `Real`: Genuine 3D CT scan of the patient
   - `Blank`: Exact zero-tensor `torch.zeros_like(real_vol)`
   - `Swap`: Genuine donor CT scan from a different real patient
   - `Shuffle Slices`: Real patient CT with permuted z-axis slices
   - `Shuffle Options`: Real patient CT with permuted answer choices
   - `Text Only`: Pure text prompt floor (no visual tokens)

In [ ]:
# =====================================================================
# CELL 1: Install Dependencies (Synced for Python 3.13 ABI)
# =====================================================================
!pip install -q -U numpy scipy transformers accelerate timm einops monai

In [ ]:
# =====================================================================
# CELL 2: Configuration, Manifests & Real CT Fail-Fast Verification
# =====================================================================
import glob, json, os, time, sys, math, copy
import numpy as np
import pandas as pd
import torch

MODE            = 'full'    # 'pilot' = 150 questions | 'full' = 7,746 questions
MODEL_ID        = 'GoodBaiBai88/M3D-LaMed-Phi-3-4B'
CHECKPOINT_STEP = 25 if MODE == 'pilot' else 250
SEED            = 42
output_dir      = f'real_ct_audit_results_{MODE}'
os.makedirs(output_dir, exist_ok=True)

np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    print(f'[GPU 0] {torch.cuda.get_device_name(0)}')
    if torch.cuda.device_count() > 1:
        print(f'[GPU 1] {torch.cuda.get_device_name(1)}')
else:
    raise RuntimeError('No GPU found — Please set Accelerator to GPU T4 x2 or T4 x1')

# ── Global Scan Discovery (O(1) Cached Lookup) ───────────────────────
print('[1/3] Indexing real 3D CT scan files on disk...')
SCAN_PATH_CACHE = {}
for p in glob.glob('/kaggle/input/**/*.npy', recursive=True) + glob.glob('**/*.npy', recursive=True):
    bn = os.path.splitext(os.path.basename(p))[0]
    SCAN_PATH_CACHE[bn] = p

if not SCAN_PATH_CACHE:
    raise FileNotFoundError(
        '[FATAL STOP] No .npy CT scan files found!\n'
        'Make sure the "medi data 3d" dataset is attached to this Kaggle notebook.\n'
        'Execution stopped to prevent dummy evaluation.'
    )
print(f'[OK] Discovered {len(SCAN_PATH_CACHE)} genuine CT scans in attached dataset.')

# ── Manifest Discovery & Smart Self-Generation ────────────────────────
print('[2/3] Resolving benchmark questions...')
def find_json(fname):
    for pat in [f'/kaggle/input/**/{fname}', f'**/{fname}', f'/kaggle/working/**/{fname}']:
        m = glob.glob(pat, recursive=True)
        if m: return m[0]
    return None

p_real  = find_json(f'real_ct_{MODE}.json') or (find_json('real_ct_full.json') if MODE == 'full' else find_json('real_ct_pilot.json'))
p_swap  = find_json(f'real_ct_swap_{MODE}.json') or (find_json('real_ct_swap_full.json') if MODE == 'full' else find_json('real_ct_swap.json'))
p_shuff = find_json(f'real_ct_shuffled_{MODE}.json') or (find_json('real_ct_shuffled_full.json') if MODE == 'full' else find_json('real_ct_shuffled.json'))

if p_real and p_swap and p_shuff:
    print(f'[Manifests] Found pre-indexed JSON files on disk: {p_real}')
    with open(p_real)  as f: real_items  = json.load(f)
    with open(p_swap)  as f: swap_items  = json.load(f)
    with open(p_shuff) as f: shuff_items = json.load(f)
else:
    print('[Manifests] No separate JSON manifest uploaded — Generating directly from medi data 3d CSVs in 2 seconds...')
    csv_files = sorted(glob.glob('/kaggle/input/**/test/**/*.csv', recursive=True)) or sorted(glob.glob('**/test/**/*.csv', recursive=True))
    if not csv_files:
        raise FileNotFoundError(
            '[FATAL ERROR] Neither pre-built manifest JSONs nor medi data 3d test CSVs found!\n'
            'Please attach kaggle_partB_audit.zip OR ensure "medi data 3d" dataset is attached.'
        )
    
    all_candidates = []
    for f in csv_files:
        task = os.path.basename(os.path.dirname(f))
        df = pd.read_csv(f)
        for _, row in df.iterrows():
            vn = str(row['VolumeName'])
            bn = os.path.splitext(os.path.splitext(vn)[0])[0]
            if bn in SCAN_PATH_CACHE:
                q_text = str(row['Question']).strip()
                ans = str(row['AnswerChoice']).strip()
                choices = {'A': str(row['Choice A']).strip(), 'B': str(row['Choice B']).strip()}
                if pd.notna(row.get('Choice C')) and str(row.get('Choice C')).strip() not in ('-', '', 'nan'):
                    choices['C'] = str(row['Choice C']).strip()
                    choices['D'] = str(row['Choice D']).strip()
                if ans in choices:
                    all_candidates.append({
                        'task': task, 'scan_id': bn, 'volume_filename': f'{bn}.npy',
                        'question': q_text, 'choices': choices, 'answer_choice': ans,
                        'answer_text': str(row['Answer']).strip()
                    })
    
    if MODE == 'pilot':
        quota = {'Task4_Existence_Detection': 70, 'Task5_Static_Temporal_Diagnosis': 40, 'Task6_Longitudinal_Temporal_Diagnosis': 40}
        real_items = []
        for task, count in quota.items():
            sub = [x for x in all_candidates if x['task'] == task]
            np.random.shuffle(sub)
            real_items.extend(sub[:count])
        np.random.shuffle(real_items)
    else:
        task5 = [x for x in all_candidates if x['task'] == 'Task5_Static_Temporal_Diagnosis']
        task6 = [x for x in all_candidates if x['task'] == 'Task6_Longitudinal_Temporal_Diagnosis']
        task4 = [x for x in all_candidates if x['task'] == 'Task4_Existence_Detection']
        np.random.shuffle(task4)
        real_items = task5 + task6 + task4[:2000]
        np.random.shuffle(real_items)
    
    for i, it in enumerate(real_items):
        it['question_id'] = f'{MODE}_{i+1:05d}'
    
    donor_pool = sorted(list(SCAN_PATH_CACHE.keys()))
    swap_items = copy.deepcopy(real_items)
    shuff_items = copy.deepcopy(real_items)
    
    for item in swap_items:
        orig = item['scan_id']
        donor = np.random.choice(donor_pool)
        while donor == orig and len(donor_pool) > 1:
            donor = np.random.choice(donor_pool)
        item['donor_scan_id'] = donor
        item['donor_volume_filename'] = f'{donor}.npy'
    
    for item in shuff_items:
        item['shuffle_slices'] = True

print(f'[OK] Successfully loaded {len(real_items)} questions for Genuine CT Audit (MODE={MODE}).')

# ── [3/3] FAIL-FAST TEST: Verify that real CT scans exist in dataset ───
first_scan_id = real_items[0]['scan_id']
sample_path = SCAN_PATH_CACHE[first_scan_id]
sample_vol = np.load(sample_path)
assert sample_vol is not None and sample_vol.size > 0, 'Corrupt scan file!'
assert not np.allclose(sample_vol, 0), 'Scan is all-zero!'
assert float(np.std(sample_vol)) > 1e-4, 'Scan is flat dummy noise!'
print(f'[VERIFIED] Real CT scan validated: {sample_path}')
print(f'           Shape: {sample_vol.shape} | Std: {sample_vol.std():.3f} | Min: {sample_vol.min():.3f} | Max: {sample_vol.max():.3f}')


In [ ]:
# =====================================================================
# CELL 3: Load Model & Tokenizer (With Multi-GPU Alignment Patch)
# =====================================================================
import os, sys, glob, re, types

# Bypass object-detection loss scipy dependency in transformers
import transformers
if hasattr(transformers, 'utils'):
    if hasattr(transformers.utils, 'import_utils'):
        transformers.utils.import_utils.is_scipy_available = lambda: False
    transformers.utils.is_scipy_available = lambda: False

from huggingface_hub import hf_hub_download
from transformers import AutoConfig, AutoTokenizer, AutoModelForCausalLM

print('[1/3] Downloading model code files...')
AutoConfig.from_pretrained(MODEL_ID, trust_remote_code=True)
hf_hub_download(repo_id=MODEL_ID, filename='modeling_m3d_lamed.py')

print('[2/3] Patching pos_embed and cross-GPU alignment directly in cache...')
cache_root = os.path.expanduser('~/.cache/huggingface')
for p in glob.glob(f'{cache_root}/**/modeling_m3d_lamed.py', recursive=True):
    with open(p, 'r', encoding='utf-8') as f: src = f.read()
    new_src = src
    if 'pos_embed=pos_embed,' in new_src:
        new_src = new_src.replace('pos_embed=pos_embed,', 'proj_type=pos_embed,')
    if 'inputs_embeds = self.get_model().embed_tokens(input_ids)' in new_src and 'image_features.to(inputs_embeds.device)' not in new_src:
        new_src = new_src.replace('inputs_embeds = self.get_model().embed_tokens(input_ids)', 'inputs_embeds = self.get_model().embed_tokens(input_ids)\n            image_features = image_features.to(inputs_embeds.device)')
    if new_src != src:
        with open(p, 'w', encoding='utf-8') as f: f.write(new_src)
        print(f'[Shield] Patched cache file: {p}')

print('[3/3] Loading model weights to GPU...')
tok = AutoTokenizer.from_pretrained(
    MODEL_ID, model_max_length=512, padding_side='right',
    use_fast=False, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16,
    device_map='auto', trust_remote_code=True)
model.eval()

# Dynamic multi-GPU alignment guarantee
def _safe_prepare_inputs(self, input_ids, position_ids, attention_mask, past_key_values, labels, images):
    vision_tower = self.get_vision_tower()
    if vision_tower is None or images is None or input_ids.shape[1] == 1:
        return input_ids, position_ids, attention_mask, past_key_values, None, labels
    else:
        image_features = self.encode_images(images)
        inputs_embeds = self.get_model().embed_tokens(input_ids)
        image_features = image_features.to(inputs_embeds.device)
        inputs_embeds = torch.cat(
            (inputs_embeds[:, :1, :], image_features, inputs_embeds[:, (image_features.shape[1] + 1):, :]), dim=1)
    return None, position_ids, attention_mask, past_key_values, inputs_embeds, labels

model.prepare_inputs_for_multimodal = types.MethodType(_safe_prepare_inputs, model)

vram = torch.cuda.memory_allocated() / (1024**3)
print(f'[model] Successfully loaded! Allocated VRAM: {vram:.2f} GB')


In [ ]:
# =====================================================================
# CELL 4: Real CT Volume Loader & Inference Engine (Strict Fail-Fast)
# =====================================================================
PROJ_OUT_NUM = 256
IMAGE_TOKENS = '<im_patch>' * PROJ_OUT_NUM

# Cache discovered scan paths for high-speed loading
SCAN_PATH_CACHE = {}

def load_real_ct_volume(scan_id):
    """Loads genuine 3D CT volume from medi data 3d with strict validation."""
    if scan_id in SCAN_PATH_CACHE:
        fpath = SCAN_PATH_CACHE[scan_id]
    else:
        matches = glob.glob(f'/kaggle/input/**/{scan_id}*.npy', recursive=True) or glob.glob(f'**/{scan_id}*.npy', recursive=True)
        if not matches:
            raise FileNotFoundError(
                f'[FATAL AUDIT ERROR] Real CT scan file for {scan_id} NOT found!\n'
                f'Execution halted immediately. No dummy values allowed.'
            )
        fpath = matches[0]
        SCAN_PATH_CACHE[scan_id] = fpath

    arr = np.load(fpath)
    
    # Strict authenticity assertions
    if arr is None or arr.size == 0:
        raise ValueError(f'Volume {scan_id} is empty!')
    if np.allclose(arr, 0):
        raise ValueError(f'Volume {scan_id} is all zeros (invalid real scan)!')
    if float(np.std(arr)) < 1e-4:
        raise ValueError(f'Volume {scan_id} lacks spatial variance (possible flat dummy)!')

    # Convert to 5D: (batch=1, channel=1, depth=32, height=256, width=256)
    if arr.ndim == 3:
        arr = arr[np.newaxis, np.newaxis, ...]
    elif arr.ndim == 4 and arr.shape[0] == 1:
        arr = arr[np.newaxis, ...]
    elif arr.ndim == 4:
        arr = arr[:, np.newaxis, ...]

    v_dev = next(model.get_model().get_vision_tower().parameters()).device
    return torch.from_numpy(arr).to(dtype=torch.float16, device=v_dev)

def format_prompt(item, include_image=True):
    q_text  = item['question']
    choices = item['choices']
    letters = sorted(choices.keys())
    opts    = '\n'.join(f'({L}) {choices[L]}' for L in letters)
    pfx     = IMAGE_TOKENS if include_image else ''
    return f'{pfx}Question: {q_text}\nOptions:\n{opts}\nAnswer: (', letters

@torch.no_grad()
def score_item(prompt, letters, gold, vol):
    t_dev = model.get_model().embed_tokens.weight.device
    ids = tok(prompt, return_tensors='pt').input_ids.to(t_dev)
    
    out = model(input_ids=ids, images=vol) if vol is not None \
          else model(input_ids=ids, images=None)
          
    lgt = out.logits[0, -1, :]
    lids = [tok.encode(L, add_special_tokens=False)[-1] for L in letters]
    cand_logits = lgt[lids].float().cpu().numpy()
    raw = {L: float(cand_logits[i]) for i, L in enumerate(letters)}
    ex = np.exp(cand_logits - np.max(cand_logits))
    lp = np.log(ex / ex.sum())
    pred = letters[int(np.argmax(lp))]
    return pred, int(pred == gold), float(lp[letters.index(gold)]), raw

print('[OK] Real CT Loader & Scoring Engine ready.')

In [ ]:
# =====================================================================
# CELL 5: Run Genuine 3D CT Audit Loop (Zero Dummy)
# =====================================================================
ckpt_path = os.path.join(output_dir, f'm3d_{MODE}_real_ct_checkpoint.csv')
records, start_idx = [], 0

if os.path.exists(ckpt_path):
    _ck       = pd.read_csv(ckpt_path)
    records   = _ck.to_dict('records')
    start_idx = len(records)
    print(f'[RESUME] Resuming from item {start_idx}/{len(real_items)}')

t0, total = time.time(), len(real_items)
print(f'[*] Starting Genuine CT Audit on {total} questions across 6 conditions...')

for idx in range(start_idx, total):
    ri  = real_items[idx]
    swi = swap_items[idx]
    si  = shuff_items[idx]

    gold    = ri['answer_choice']
    gold_so = si['answer_choice']

    # 1. REAL: Load genuine patient 3D CT scan
    V_real = load_real_ct_volume(ri['scan_id'])
    p_r, L_r = format_prompt(ri, True)
    pr, cr, lpr, lr = score_item(p_r, L_r, gold, V_real)

    # 2. BLANK: Exact all-zeros volume matching real CT dimensions
    V_blank = torch.zeros_like(V_real)
    pb, cb, lpb, lb = score_item(p_r, L_r, gold, V_blank)

    # 3. SWAP: Genuine donor patient 3D CT scan
    V_swap = load_real_ct_volume(swi['donor_scan_id'])
    ps, cs, lps, ls = score_item(p_r, L_r, gold, V_swap)

    # 4. SHUFFLE SLICES: Permute z-slices of real CT scan
    perm = torch.randperm(32)
    V_sl = V_real[:, :, perm, :, :].contiguous()
    psl, csl, lpsl, lsl = score_item(p_r, L_r, gold, V_sl)

    # 5. SHUFFLE OPTIONS: Real CT scan with permuted question choices
    p_so, L_so = format_prompt(si, True)
    pso, cso, lpso, lso = score_item(p_so, L_so, gold_so, V_real)

    # 6. TEXT ONLY: Question prompt without image tokens
    p_to, L_to = format_prompt(ri, False)
    pto, cto, lpto, lto = score_item(p_to, L_to, gold, None)

    records.append({
        'question_id':         ri['question_id'],
        'task':                ri['task'],
        'scan_id':             ri['scan_id'],
        'donor_scan_id':       swi['donor_scan_id'],
        'gold_answer':         gold,
        'pred_real':           pr,  'pred_blank':          pb,
        'pred_swap':           ps,  'pred_shuf_slices':    psl,
        'pred_shuf_opts':      pso, 'pred_textonly':       pto,
        'corr_real':           cr,  'corr_blank':          cb,
        'corr_swap':           cs,  'corr_shuf_slices':    csl,
        'corr_shuf_opts':      cso, 'corr_textonly':       cto,
        'lp_real':             lpr, 'lp_blank':            lpb,
        'lp_swap':             lps, 'lp_shuf_slices':      lpsl,
        'lp_shuf_opts':        lpso,'lp_textonly':         lpto,
        'delta_logp':          lpr - lpb,
        'logits_real':         json.dumps(lr),
        'logits_blank':        json.dumps(lb),
        'logits_swap':         json.dumps(ls),
        'logits_shuf_slices':  json.dumps(lsl),
        'logits_shuf_opts':    json.dumps(lso),
        'logits_textonly':     json.dumps(lto),
    })

    if (idx + 1) % CHECKPOINT_STEP == 0 or (idx + 1) == total:
        elapsed = time.time() - t0
        done    = idx + 1 - start_idx
        rate    = done / max(elapsed, 1e-4)
        eta     = (total - idx - 1) / max(rate, 1e-4) / 60
        print(f'[{idx+1}/{total}] {elapsed:.0f}s elapsed | {rate:.2f} it/s | ETA {eta:.1f}m')
        pd.DataFrame(records).to_csv(ckpt_path, index=False)

df = pd.DataFrame(records)
csv_path = os.path.join(output_dir, f'm3d_{MODE}_real_ct_per_item.csv')
df.to_csv(csv_path, index=False)
print(f'\n[DONE] Genuine CT Audit complete! Saved {len(df)} items to {csv_path}')

In [ ]:
# =====================================================================
# CELL 6: Calculate Summary Table & Statistical Tests
# =====================================================================
from IPython.display import display
import math

summary = []
groups = [('overall', df)] + [(t, df[df['task'] == t]) for t in df['task'].unique() if t]

for name, sub in groups:
    n = len(sub)
    if n == 0: continue
    
    acc_r  = sub['corr_real'].mean()
    acc_b  = sub['corr_blank'].mean()
    acc_sw = sub['corr_swap'].mean()
    acc_sl = sub['corr_shuf_slices'].mean()
    acc_so = sub['corr_shuf_opts'].mean()
    acc_to = sub['corr_textonly'].mean()
    
    # Exact Binomial McNemar (Real vs Blank)
    b = int(((sub['corr_real'] == 1) & (sub['corr_blank'] == 0)).sum())
    c = int(((sub['corr_real'] == 0) & (sub['corr_blank'] == 1)).sum())
    nd = b + c
    p_mcn = 1.0 if nd == 0 else min(1.0, 2.0 * sum(math.comb(nd, i) for i in range(min(b, c) + 1)) / (2**nd))
    
    summary.append({
        'task':              name,
        'n':                 n,
        'acc_real_ct':       round(acc_r, 4),
        'acc_blank_ct':      round(acc_b, 4),
        'acc_swap_donor':    round(acc_sw, 4),
        'acc_shuf_slices':   round(acc_sl, 4),
        'acc_shuf_opts':     round(acc_so, 4),
        'acc_text_only':     round(acc_to, 4),
        'delta_blank':       round(acc_r - acc_b, 4),
        'p_mcnemar_blank':   round(p_mcn, 6),
        'mean_delta_logp':   round(sub['delta_logp'].mean(), 4),
    })

summary_df = pd.DataFrame(summary)
summary_path = os.path.join(output_dir, f'm3d_{MODE}_real_ct_summary.csv')
summary_df.to_csv(summary_path, index=False)

print('=' * 95)
print(f'M3D-LAMED GENUINE 3D CT AUDIT SUMMARY ({MODE.upper()} RUN):')
display(summary_df)
print('=' * 95)

In [ ]:
# =====================================================================
# CELL 7: Zip & One-Click Download
# =====================================================================
import shutil
from IPython.display import FileLink, display

zip_name = f'm3d_{MODE}_real_ct_audit_results'
zip_path = f'/kaggle/working/{zip_name}'
shutil.make_archive(zip_path, 'zip', output_dir)
print(f'[ZIP Complete] Created {zip_name}.zip successfully!')
display(FileLink(f'{zip_name}.zip'))